# Week 11, Day 6: Forge: Week 11 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w011/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1. amplitude", round(np.hypot(1, -1), 4), " phase", round(np.arctan2(-1, 1), 4), "rad: cos t - sin t = sqrt(2) cos(t + pi/4)")
print("2. resolution", 1 / 4, "Hz, Nyquist", 1000 / 2, "Hz")
print("3. gain", round(1 / np.sqrt(2), 4), "(and a phase lag of 45 degrees)")

In [ ]:
rng = np.random.default_rng(1993)
fs, T = 1000, 20
N = fs * T
t = np.arange(N) / fs

def aperiodic(n, chi):
    X = np.fft.rfft(rng.normal(size=n)); f = np.fft.rfftfreq(n, 1 / fs)
    s = np.zeros_like(f); s[1:] = f[1:] ** (-chi / 2)
    y = np.fft.irfft(X * s, n); return y / y.std()

_f_inst = 8 + np.convolve(rng.normal(size=N), np.ones(500) / 500, "same") * 15      # wandering theta frequency
_theta_phase = np.angle(np.exp(1j * np.cumsum(2 * np.pi * _f_inst / fs)))
_gamma = 0.25 * (1 + 0.8 * np.cos(_theta_phase)) * np.cos(2 * np.pi * 60 * t)
lfp = np.cos(_theta_phase) + _gamma + 0.8 * aperiodic(N, 2.0)

position = (t % 4) / 4                                   # 0 to 1 along the track on each lap
_field = np.exp(-(position - 0.5) ** 2 / (2 * 0.12 ** 2))
_through = np.clip((position - 0.2) / 0.6, 0, 1)
_pref = 2 * np.pi * (0.9 - 0.7 * _through)               # hidden: phase falls by 0.7 cycles across the field
spikes = np.nonzero(rng.random(N) < 0.04 * _field * np.exp(2 * np.cos(_theta_phase - _pref)) / 2.3)[0]

def welch(x, fs, nperseg):
    w = np.hanning(nperseg); out = []
    for s in range(0, len(x) - nperseg + 1, nperseg // 2):
        seg = x[s:s + nperseg] - x[s:s + nperseg].mean()
        p = np.abs(np.fft.rfft(w * seg)) ** 2 / (fs * np.sum(w ** 2)); p[1:-1] *= 2; out.append(p)
    return np.fft.rfftfreq(nperseg, 1 / fs), np.mean(out, axis=0)

def bandpass(x, fs, lo, hi, edge=1.0):
    X = np.fft.rfft(x); f = np.fft.rfftfreq(len(x), 1 / fs)
    d = np.maximum(lo - f, 0) + np.maximum(f - hi, 0)
    return np.fft.irfft(X * np.exp(-d ** 2 / (2 * edge ** 2)), len(x))

def analytic_signal(x):
    X = np.fft.fft(x); h = np.zeros(len(x)); h[0] = 1; h[1:len(x) // 2] = 2
    if len(x) % 2 == 0: h[len(x) // 2] = 1
    return np.fft.ifft(X * h)

print(len(spikes), "spikes from the place cell")
plt.figure(figsize=(7, 2.5)); plt.plot(t[:2000], lfp[:2000], lw=0.7); plt.xlabel("time (s)"); plt.title("two seconds of LFP"); plt.show()

### Exercise: Part A: theta above the background

Compute the Welch spectrum with 2048-sample segments. Fit the aperiodic background as a line on log-log axes over 2 to 100 Hz, excluding 4 to 12 Hz and 50 to 70 Hz, and store its exponent in `chi`. Find the frequency between 4 and 12 Hz where log10 PSD rises furthest above the fit (`theta_peak`) and the height there (`theta_excess`).

In [ ]:
f, p = welch(lfp, fs, 2048)
sel = (f >= 2) & (f <= 100) & ~((f >= 4) & (f <= 12)) & ~((f >= 50) & (f <= 70))
slope, icpt = ...
chi = -slope
excess = np.log10(p[1:]) - (icpt + slope * np.log10(f[1:]))
band = (f[1:] >= 4) & (f[1:] <= 12)
theta_peak = ...
theta_excess = ...

plt.figure(figsize=(5, 3.2)); plt.loglog(f[1:], p[1:]); plt.loglog(f[1:], 10 ** (icpt + slope * np.log10(f[1:])), "--")
plt.xlim(1, 200); plt.xlabel("frequency (Hz)"); plt.ylabel("PSD"); plt.show()
print(f"aperiodic exponent {chi:.2f}; theta peak at {theta_peak:.2f} Hz, {theta_excess:.2f} log10 units above the background")

In [ ]:
import numpy as np
def test_exponent():
    "chi is within 0.3 of the true exponent 2"
    assert abs(chi - 2.0) < 0.3

def test_peak():
    "theta_peak is the frequency between 4 and 12 Hz with the largest excess over the fit, between 7 and 9 Hz"
    assert 7 <= theta_peak <= 9

def test_excess():
    "theta rises more than 0.5 log10 units above the background"
    assert theta_excess > 0.5

test_exponent()
test_peak()
test_excess()
print("All checks passed")

### Exercise: Part B: phase-amplitude coupling

Get theta phase (band-pass 6 to 10 Hz, analytic signal) and the gamma envelope (band-pass 50 to 70 Hz with 3 Hz edges, magnitude of the analytic signal). Write `modulation_index(phase, amp, nbins=18)`, compute `mi_real`, and build a null distribution `mi_null` from 50 circular shifts of the envelope by random offsets of at least one second. Store the centre of the theta-phase bin with the largest mean gamma in `peak_phase`.

In [ ]:
theta_phase = np.angle(analytic_signal(bandpass(lfp, fs, 6, 10)))
gamma_amp = np.abs(analytic_signal(bandpass(lfp, fs, 50, 70, edge=3.0)))

def modulation_index(phase, amp, nbins=18):
    bins = np.clip(((phase + np.pi) / (2 * np.pi) * nbins).astype(int), 0, nbins - 1)
    mean_amp = np.array([amp[bins == b].mean() for b in range(nbins)])
    P = mean_amp / mean_amp.sum()
    return ...

mi_real = modulation_index(theta_phase, gamma_amp)
mi_null = [...]
centres = -np.pi + (np.arange(18) + 0.5) * 2 * np.pi / 18
bins = np.clip(((theta_phase + np.pi) / (2 * np.pi) * 18).astype(int), 0, 17)
profile = np.array([gamma_amp[bins == b].mean() for b in range(18)])
peak_phase = ...

plt.figure(figsize=(5, 3)); plt.bar(np.degrees(centres), profile, width=18); plt.xlabel("theta phase (deg)"); plt.ylabel("mean gamma envelope"); plt.show()
print(f"MI = {mi_real:.4f};  largest of 50 shifted controls = {max(mi_null):.4f};  gamma peaks at theta phase {np.degrees(peak_phase):.0f} deg")

In [ ]:
import numpy as np
def test_mi():
    "modulation_index is 0 for a flat profile and positive otherwise"
    ph = np.linspace(-np.pi, np.pi, 1800, endpoint=False)
    assert np.isclose(modulation_index(ph, np.ones(1800)), 0, atol=1e-12)
    assert modulation_index(ph, 1 + 0.5 * np.cos(ph)) > 0.01

def test_coupling():
    "the real MI exceeds every one of 50 shifted controls"
    assert mi_real > max(mi_null)

def test_phase():
    "gamma is strongest near theta phase 0 (the theta peak)"
    assert abs(np.angle(np.exp(1j * peak_phase))) < 0.6

test_mi()
test_coupling()
test_phase()
print("All checks passed")

### Exercise: Part C: phase precession

Keep spikes with position between 0.2 and 0.8 (the field), and store their theta phases in `spike_phase` and their position within the field, rescaled to 0 to 1, in `spike_pos`. Fit the circular-linear relation phase = 2π·slope·position + offset by searching slopes from −1.5 to 1.5 cycles per field for the one that maximises the mean resultant length of the residual phases (Kempter and colleagues, 2012). Store it in `best_slope`.

In [ ]:
in_field = spikes[(position[spikes] > 0.2) & (position[spikes] < 0.8)]
spike_phase = ...
spike_pos = ...

slopes = np.linspace(-1.5, 1.5, 301)
fit_R = [np.abs(np.mean(np.exp(1j * (spike_phase - 2 * np.pi * s * spike_pos)))) for s in slopes]
best_slope = ...

plt.figure(figsize=(5, 3.5))
for shift in [0, 360]:
    plt.plot(spike_pos, np.degrees(spike_phase) % 360 + shift, "k.", ms=3)
plt.xlabel("position through the field"); plt.ylabel("theta phase (deg), plotted twice"); plt.show()
print(f"{len(spike_phase)} spikes in the field;  precession slope {best_slope:.2f} cycles per field")

In [ ]:
import numpy as np
def test_inputs():
    "spike_phase and spike_pos hold the theta phase and normalised field position of in-field spikes"
    assert len(spike_phase) == len(spike_pos) > 100
    assert np.all((spike_pos >= 0) & (spike_pos <= 1))

def test_slope():
    "the fitted precession slope is within 0.15 cycles of the true -0.7 cycles per field"
    assert abs(best_slope + 0.7) < 0.15

test_inputs()
test_slope()
print("All checks passed")

### Exercise: Part D: write the FFT

Implement the radix-2 FFT recursively. For length 1 return the input. Otherwise transform the even-indexed and odd-indexed samples separately (E and O), and combine them with the twiddle factors $w_k = e^{-i2\pi k/N}$: $X_k = E_k + w_kO_k$ and $X_{k + N/2} = E_k - w_kO_k$ for k from 0 to N/2 − 1. The derivation set shows why.

In [ ]:
def my_fft(x):
    x = np.asarray(x, dtype=complex)
    N = len(x)
    if N == 1:
        return x
    E = my_fft(x[0::2])
    O = my_fft(x[1::2])
    w = np.exp(-2j * np.pi * np.arange(N // 2) / N)
    return ...

x = np.random.default_rng(0).normal(size=1024)
print("matches numpy:", np.allclose(my_fft(x), np.fft.fft(x)))

In [ ]:
import numpy as np
def test_match():
    "my_fft matches np.fft.fft for powers of two"
    for n in [1, 2, 8, 64, 256]:
        x = np.random.default_rng(n).normal(size=n) + 1j * np.random.default_rng(n + 1).normal(size=n)
        assert np.allclose(my_fft(x), np.fft.fft(x))

test_match()
print("All checks passed")